In [2]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

2026-05-10 03:44:18.485114: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778384658.693856      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778384658.752208      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778384659.239753      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778384659.239798      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778384659.239801      57 computation_placer.cc:177] computation placer alr

In [3]:
# ============================================
# LOAD CIFAR-10 DATASET (TESTING)
# ============================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("CIFAR Test Shape:", x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
CIFAR Test Shape: (10000, 32, 32, 3)


In [4]:
# ============================================
# NORMALIZE CIFAR TEST DATA
# ============================================

x_test = x_test.astype("float32") / 255.0

y_test = y_test.flatten()

In [5]:
# ============================================
# LOAD CINIC-10 TRAIN DATASET
# ============================================

cinic_train = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/mengcius/cinic10/train',
    image_size=(32,32),
    batch_size=64,
    shuffle=True
)

Found 90000 files belonging to 10 classes.


I0000 00:00:1778384796.456687      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778384796.462603      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [6]:
# ============================================
# NORMALIZE CINIC TRAIN DATA
# ============================================

cinic_train = cinic_train.map(
    lambda x, y: (
        tf.cast(x, tf.float32) / 255.0,
        tf.cast(y, tf.int32)
    )
)

In [7]:
# ============================================
# DATA AUGMENTATION
# ============================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [8]:
# ============================================
# DENSE BLOCK
# ============================================

def dense_block(x, num_layers, growth_rate):

    for _ in range(num_layers):

        y = layers.BatchNormalization()(x)
        y = layers.ReLU()(y)

        y = layers.Conv2D(
            4 * growth_rate,
            (1,1),
            padding='same',
            use_bias=False
        )(y)

        y = layers.BatchNormalization()(y)
        y = layers.ReLU()(y)

        y = layers.Conv2D(
            growth_rate,
            (3,3),
            padding='same',
            use_bias=False
        )(y)

        x = layers.Concatenate()([x, y])

    return x

In [9]:
# ============================================
# TRANSITION LAYER
# ============================================

def transition_layer(x, reduction=0.5):

    filters = int(x.shape[-1] * reduction)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        use_bias=False
    )(x)

    x = layers.AveragePooling2D((2,2), strides=2)(x)

    return x

In [10]:
# ============================================
# BUILD DENSENET MODEL
# ============================================

def build_densenet(input_shape=(32,32,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)

    x = layers.Conv2D(
        64,
        (3,3),
        padding='same',
        use_bias=False
    )(x)

    x = dense_block(x, num_layers=4, growth_rate=12)
    x = transition_layer(x)

    x = dense_block(x, num_layers=4, growth_rate=12)
    x = transition_layer(x)

    x = dense_block(x, num_layers=4, growth_rate=12)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation='softmax'
    )(x)

    model = models.Model(inputs, outputs)

    return model

In [11]:
# ============================================
# CREATE MODEL
# ============================================

model = build_densenet()

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32, 32, 3) │          0 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │      1,728 │ sequential[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │      3,072 │ re_lu[0][0]       │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        192 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │      5,184 │ re_lu_1[0][0]     │
│                     │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32, 32,    │          0 │ conv2d[0][0],     │
│ (Concatenate)       │ 76)               │            │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        304 │ concatenate[0][0] │
│ (BatchNormalizatio… │ 76)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 76)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │      3,648 │ re_lu_2[0][0]     │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        192 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │      5,184 │ re_lu_3[0][0]     │
│                     │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 32, 32,    │          0 │ concatenate[0][0

 Total params: 140,410 (548.48 KB)

 Trainable params: 136,818 (534.45 KB)

 Non-trainable params: 3,592 (14.03 KB)

In [12]:
# ============================================
# COMPILE MODEL
# ============================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
for images, labels in cinic_train.take(1):
    print(images.shape)
    print(labels[:10])

(64, 32, 32, 3)
tf.Tensor([8 3 2 9 5 7 3 3 9 7], shape=(10,), dtype=int32)


In [14]:
# ============================================
# TRAIN ON CINIC-10
# ============================================

history = model.fit(
    cinic_train,
    epochs=40,
    validation_data=(x_test, y_test)
)

Epoch 1/40


I0000 00:00:1778384815.148126     140 cuda_dnn.cc:529] Loaded cuDNN version 91002


1407/1407 ━━━━━━━━━━━━━━━━━━━━ 183s 117ms/step - accuracy: 0.3243 - loss: 1.8089 - val_accuracy: 0.4778 - val_loss: 1.4924
Epoch 2/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 113s 80ms/step - accuracy: 0.4724 - loss: 1.4329 - val_accuracy: 0.4325 - val_loss: 1.7042
Epoch 3/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 113s 80ms/step - accuracy: 0.5251 - loss: 1.3072 - val_accuracy: 0.5658 - val_loss: 1.2509
Epoch 4/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 113s 80ms/step - accuracy: 0.5548 - loss: 1.2368 - val_accuracy: 0.5518 - val_loss: 1.2867
Epoch 5/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 113s 80ms/step - accuracy: 0.5746 - loss: 1.1786 - val_accuracy: 0.5241 - val_loss: 1.5419
Epoch 6/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 113s 80ms/step - accuracy: 0.5940 - loss: 1.1338 - val_accuracy: 0.6287 - val_loss: 1.0347
Epoch 7/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 113s 80ms/step - accuracy: 0.6098 - loss: 1.0932 - val_accuracy: 0.6726 - val_loss: 0.9481
Epoch 8/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 113s 80ms/step - accuracy: 0.6167 - l

In [15]:
# ============================================
# TEST ON CIFAR-10
# ============================================

test_loss, test_acc = model.evaluate(x_test, y_test)

print("CINIC → CIFAR DenseNet Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.8052 - loss: 0.5764
CINIC → CIFAR DenseNet Accuracy: 0.8066999912261963


In [16]:
# ============================================
# SAVE MODEL
# ============================================

model.save('/kaggle/working/cinic_to_cifar_densenet.keras')